# Jev routes, Claude answers, humans decide

An engineering operations assistant gets a mix of requests. Most are routine, a few are big calls, and a few are
one-way doors that no model should make alone. Before any Claude call, Jev reads the request and returns a typed
decision with a confidence, and a Strands hook sends it to the right place:

| Tier | Route | Example |
|---|---|---|
| General question | Claude Sonnet 5 | "How do I rotate an IAM access key?" |
| Big decision, reversible | Claude Opus 5 | "Roll back this release or hotfix forward?" |
| One-way door: payments, contracts, irreversible | a human | "Pick a vendor for a 3-year, $2M contract" |

| # | Rule | Route |
|---|---|---|
| 1 | Jev flags a one-way door | human, whatever the confidence |
| 2 | confidence below 50% | human |
| 3 | confidence 50% to 70% | Opus 5 |
| 4 | confidence 70% or more | the tier Jev picked |

Jev is called once per request, and the answers are cached in `data/jev_answers.json`, so every later cell and
every rerun reuses them.

In [ ]:
import json
import os
import statistics
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv
from strands import Agent
from strands.hooks import AfterInvocationEvent, BeforeInvocationEvent, HookProvider, HookRegistry
from strands.models import BedrockModel
from typesafe_sdk import Choice, Noul, TypeSafeClient

from request_set import REQUESTS, TIERS, Request

load_dotenv(".env")

REGION = os.getenv("AWS_REGION", "us-east-1")
MODELS = {
    "sonnet": os.getenv("SONNET_MODEL_ID", "global.anthropic.claude-sonnet-5"),
    "opus": os.getenv("OPUS_MODEL_ID", "us.anthropic.claude-opus-5"),
}
USD_PER_MILLION = {"sonnet": (2.0, 10.0), "opus": (5.0, 25.0)}  # list price per million tokens, in / out
JEV_USD_PER_TOKEN = 0.042 / 1_000_000
OPUS_BELOW, HUMAN_BELOW, ONE_WAY_AT = 0.70, 0.50, 0.50
JEV_CACHE = Path("data/jev_answers.json")
SYSTEM = (
    "You are an engineering operations assistant. Answer in five sentences or fewer. "
    "For a decision, give a clear recommendation and the main reason for it."
)
BY_ID = {r.id: r for r in REQUESTS}
print(f"{len(REQUESTS)} requests:", {t: sum(r.tier == t for r in REQUESTS) for t in TIERS})

## Jev's two questions, one call per request

A three-way choice for the tier, and a separate yes/no for the one-way door. The separate question gets its own
probability, so a payment or contract decision is caught even when the three-way choice is unsure.

In [ ]:
QUESTIONS = {
    "tier": Choice(
        instructions="Which kind of request is this, for an engineering operations assistant?",
        criteria={
            "general": "a routine question: how-to, lookup, summary, explanation or draft; low stakes and easy to redo",
            "big_decision": "a consequential decision that can still be reversed: rollback, scaling, prioritising, incident response, ownership",
            "one_way_door": "an irreversible or sensitive commitment: payments, money movement, contracts, vendor spend, legal sign-off, deleting data",
        },
    ),
    "one_way_door": Noul(
        instructions="This request asks someone to make or approve a decision that is hard or impossible to reverse, "
        "or that commits money, payments, contracts or legal obligations."
    ),
}


def ask_jev(requests: list[Request], jev: TypeSafeClient | None = None) -> dict[str, dict[str, Any]]:
    """Jev's decision for every request, from the cache when it has one. Only uncached requests cost a Jev call."""
    cache = json.loads(JEV_CACHE.read_text()) if JEV_CACHE.exists() else {}
    todo = [r for r in requests if r.id not in cache]
    if todo:
        jev = jev or TypeSafeClient()

        def one(r: Request) -> tuple[str, dict[str, Any]]:
            start = time.perf_counter()
            response = jev.system_one(state={"REQUEST": r.text}, questions=QUESTIONS)
            tier = response.answers["tier"]
            return r.id, {
                "tier": tier.choice, "confidence": float(tier.confidence), "probabilities": dict(tier.probabilities),
                "one_way_door": float(response.answers["one_way_door"].noul),
                "jev_ms": round((time.perf_counter() - start) * 1000, 1), "jev_tokens": response.usage.input_tokens or 0,
            }

        with ThreadPoolExecutor(8) as pool:
            cache.update(dict(pool.map(one, todo)))
        JEV_CACHE.parent.mkdir(exist_ok=True)
        JEV_CACHE.write_text(json.dumps(cache, indent=1))
    print(f"Jev: {len(todo)} new calls, {len(requests) - len(todo)} from cache")
    return {r.id: cache[r.id] for r in requests}


def route(decision: dict[str, Any], opus_below: float = OPUS_BELOW, human_below: float = HUMAN_BELOW) -> tuple[str, str]:
    """The routing rules, in order. Returns the route (sonnet, opus or human) and the reason."""
    if decision["one_way_door"] >= ONE_WAY_AT or decision["tier"] == "one_way_door":
        return "human", "one-way door"
    if decision["confidence"] < human_below:
        return "human", f"confidence {decision['confidence']:.0%} is below {human_below:.0%}"
    if decision["confidence"] < opus_below:
        return "opus", f"confidence {decision['confidence']:.0%} is below {opus_below:.0%}"
    return ("sonnet", "general question") if decision["tier"] == "general" else ("opus", "big decision")

## The router is a Strands hook

`JevRouter` runs on `BeforeInvocationEvent`. For Sonnet or Opus it sets the agent's model for that invocation. For a
human it files an escalation ticket and cancels the invocation, so Claude is never called. `RunStats` records what
each run cost. The agent, its prompt and the calling code are the same for every route.

In [ ]:
@dataclass
class RunStats(HookProvider):
    """One request's route, model, tokens, seconds and answer, filled in by hooks."""

    request_id: str = ""
    route: str = ""
    reason: str = ""
    input_tokens: int = 0
    output_tokens: int = 0
    seconds: float = 0.0
    answer: str = ""
    started: float = field(default=0.0, repr=False)

    def register_hooks(self, registry: HookRegistry, **_: Any) -> None:
        registry.add_callback(BeforeInvocationEvent, self.start)
        registry.add_callback(AfterInvocationEvent, self.finish)

    def start(self, event: BeforeInvocationEvent) -> None:
        self.started = time.perf_counter()

    def finish(self, event: AfterInvocationEvent) -> None:
        if event.result is not None:
            usage = event.result.metrics.accumulated_usage
            self.input_tokens, self.output_tokens = usage["inputTokens"], usage["outputTokens"]
            self.answer = str(event.result).strip()
        self.seconds = time.perf_counter() - self.started

    @property
    def usd(self) -> float:
        if self.route not in USD_PER_MILLION:
            return 0.0
        usd_in, usd_out = USD_PER_MILLION[self.route]
        return (self.input_tokens * usd_in + self.output_tokens * usd_out) / 1_000_000


class JevRouter(HookProvider):
    """Before the invocation: Jev's decision picks Sonnet, Opus or a human. A human route cancels the Claude call."""

    def __init__(self, decisions: dict[str, dict[str, Any]], models: dict[str, Any]) -> None:
        self.decisions, self.models, self.tickets = decisions, models, []

    def register_hooks(self, registry: HookRegistry, **_: Any) -> None:
        registry.add_callback(BeforeInvocationEvent, self.pick)

    def pick(self, event: BeforeInvocationEvent) -> None:
        stats: RunStats = event.invocation_state["stats"]
        decision = self.decisions[stats.request_id]
        stats.route, stats.reason = route(decision)
        if stats.route == "human":
            self.tickets.append({"request": stats.request_id, "reason": stats.reason, **decision})
            stats.answer = event.cancel = f"Escalated to a human: {stats.reason}."
        else:
            event.agent.model = self.models[stats.route]


def make_models() -> dict[str, BedrockModel]:
    return {name: BedrockModel(model_id=model_id, region_name=REGION, max_tokens=2000) for name, model_id in MODELS.items()}


def answer(request: Request, router: JevRouter | None = None, fixed: str | None = None) -> RunStats:
    """One request on a fresh agent: routed by Jev, or pinned to one model for the baselines."""
    stats = RunStats(request_id=request.id)
    models = router.models if router else make_models()
    hooks = [stats] + ([router] if router else [])
    if fixed:
        stats.route, stats.reason = fixed, "baseline"
    agent = Agent(model=models[fixed or "sonnet"], system_prompt=SYSTEM, hooks=hooks, callback_handler=None)
    agent(request.text, invocation_state={"stats": stats})
    return stats

## Jev's decisions, fetched once

In [ ]:
decisions = ask_jev(REQUESTS)
router = JevRouter(decisions, make_models())
pd.DataFrame([
    {"id": r.id, "label": r.tier, "jev": d["tier"], "confidence": d["confidence"], "one_way_door": d["one_way_door"],
     "route": route(d)[0], "reason": route(d)[1]}
    for r in REQUESTS for d in [decisions[r.id]]
]).round(3).head(12)

## One request per tier, end to end

In [ ]:
for rid in ("R-01", "R-21", "R-34"):
    run = answer(BY_ID[rid], router)
    print(f"{rid}  {BY_ID[rid].text[:80]}")
    print(f"      -> {run.route} ({run.reason})   {run.seconds:.1f}s   ${run.usd:.4f}")
    print(f"      {run.answer[:300]}\n")
print("escalation queue:", json.dumps(router.tickets[-1], indent=1)[:400])

## All 40 requests, three ways
All Sonnet, all Opus, and routed by Jev. Bedrock calls run 8 at a time.

In [ ]:
def run_arm(name: str) -> list[RunStats]:
    fixed = None if name == "routed" else name
    rt = JevRouter(decisions, make_models()) if name == "routed" else None
    with ThreadPoolExecutor(8) as pool:
        return list(pool.map(lambda r: answer(r, rt, fixed), REQUESTS))


arms = {name: run_arm(name) for name in ("sonnet", "opus", "routed")}
rows = [
    {"arm": name, "id": run.request_id, "label": BY_ID[run.request_id].tier, "route": run.route,
     "seconds": run.seconds + (decisions[run.request_id]["jev_ms"] / 1000 if name == "routed" else 0.0),
     "usd": run.usd + (decisions[run.request_id]["jev_tokens"] * JEV_USD_PER_TOKEN if name == "routed" else 0.0)}
    for name, runs in arms.items() for run in runs
]
df = pd.DataFrame(rows)
len(df)

### Safety first: did any one-way door reach a model?

In [ ]:
def scorecard(frame: pd.DataFrame) -> pd.DataFrame:
    out = []
    for name, g in frame.groupby("arm", sort=False):
        doors = g[g.label == "one_way_door"]
        big = g[g.label == "big_decision"]
        out.append({
            "arm": name,
            "one-way doors answered by a model": int((doors.route != "human").sum()),
            "big decisions answered by Sonnet": int((big.route == "sonnet").sum()),
            "sent to a human": int((g.route == "human").sum()),
            "total USD": round(g.usd.sum(), 4),
            "median seconds": round(g[g.route != "human"].seconds.median(), 2),
        })
    return pd.DataFrame(out).set_index("arm")


card = scorecard(df)
card

In [ ]:
doors = card["one-way doors answered by a model"].astype(int)
saving = 1 - card.at["routed", "total USD"] / card.at["opus", "total USD"]
print(f"Routed vs all Opus: {saving:.0%} cheaper, {doors['routed']} one-way doors reached a model "
      f"(all Sonnet: {doors['sonnet']}, all Opus: {doors['opus']}), "
      f"{int(card.at['routed', 'sent to a human'])} of {len(REQUESTS)} sent to a human")
fig, ax = plt.subplots(figsize=(7, 3.5))
card["total USD"].plot.bar(ax=ax, color=["#4f8cff", "#a06bff", "#18c29c"], rot=0)
ax.set_ylabel("USD for 40 requests")
plt.tight_layout()

### Where each tier went

In [ ]:
mix = df[df.arm == "routed"].pivot_table(index="label", columns="route", values="id", aggfunc="count", fill_value=0)
mix = mix.reindex(index=list(TIERS), columns=["sonnet", "opus", "human"], fill_value=0)
print("routing accuracy against the labels:",
      f"{sum(mix.loc[t, r] for t, r in [('general', 'sonnet'), ('big_decision', 'opus'), ('one_way_door', 'human')]) / len(REQUESTS):.0%}")
mix

## Moving the thresholds, with no new Jev calls

Every route below reuses the cached Jev answers and the per-request cost already measured on each model.

In [ ]:
cost = {name: {run.request_id: run.usd for run in runs} for name, runs in arms.items() if name != "routed"}
sweep = []
for opus_below in (0.5, 0.6, 0.7, 0.8, 0.9):
    routes = {r.id: route(decisions[r.id], opus_below=opus_below)[0] for r in REQUESTS}
    sweep.append({
        "opus below": opus_below,
        "USD": sum(cost[routes[r.id]][r.id] if routes[r.id] != "human" else 0.0 for r in REQUESTS),
        "one-way doors to a model": sum(r.tier == "one_way_door" and routes[r.id] != "human" for r in REQUESTS),
        "big decisions to Sonnet": sum(r.tier == "big_decision" and routes[r.id] == "sonnet" for r in REQUESTS),
        "to a human": sum(v == "human" for v in routes.values()),
    })
pd.DataFrame(sweep).round(4).set_index("opus below")

## Read this before trusting the numbers

- 40 requests is a small, synthetic set, and the thresholds are chosen on the same set they are scored on.
- The cost figures use Anthropic list prices; Bedrock bills separately. Human review time is not priced.
- Jev's latency is measured when it is first called and added to the routed arm; cached reruns do not call it again.
- A human route cancels the Claude call entirely. In production the ticket would carry Jev's reason to the reviewer.